In [14]:
import os
import json
import time
import hashlib
import argparse
from typing import List, Dict, Any, Optional, Tuple

import redis
from opensearchpy import OpenSearch
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

#from langchain.memory import ConversationBufferMemory
#from langchain.memory.buffer import ConversationBufferMemory
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.messages import HumanMessage, AIMessage

os.environ["OPENAI_API_KEY"]=""

# -----------------------
# CONFIG
# -----------------------
INDEX_NAME = os.getenv("OPENSEARCH_INDEX", "rag_pdf_index")
TEXT_FIELD = "text"
VECTOR_FIELD = "embedding"

EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

# OpenSearch connection
OS_HOST = os.getenv("OPENSEARCH_HOST", "10.103.6.142")
OS_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OS_USER = os.getenv("OPENSEARCH_USER", "admin")
OS_PASS = os.getenv("OPENSEARCH_PASS", "Aa1Bb2Cc3Dd4")

USE_SSL = os.getenv("OPENSEARCH_USE_SSL", "true").lower() == "true"
VERIFY_CERTS = os.getenv("OPENSEARCH_VERIFY_CERTS", "false").lower() == "true"
CA_CERTS = os.getenv("OPENSEARCH_CA_CERTS", "") or None
SSL_SHOW_WARN = os.getenv("OPENSEARCH_SSL_SHOW_WARN", "false").lower() == "true"

# Redis cache
REDIS_URL = os.getenv("REDIS_URL", "redis://10.103.6.142:6379/0")
REDIS_PREFIX = os.getenv("REDIS_PREFIX", "rag:os3")

TTL_EMBEDDING = int(os.getenv("TTL_EMBEDDING", "86400"))  # 1 day
TTL_RETRIEVAL = int(os.getenv("TTL_RETRIEVAL", "300"))    # 5 min
TTL_ANSWER = int(os.getenv("TTL_ANSWER", "300"))          # 5 min

CACHE_EMBEDDING = os.getenv("CACHE_EMBEDDING", "true").lower() == "true"
CACHE_RETRIEVAL = os.getenv("CACHE_RETRIEVAL", "true").lower() == "true"
#CACHE_ANSWER = os.getenv("CACHE_ANSWER", "false").lower() == "true"
CACHE_ANSWER=True

# Memory behavior
MAX_HISTORY_TURNS = int(os.getenv("MAX_HISTORY_TURNS", "10"))  # last N turns included
MAX_HISTORY_CHARS = int(os.getenv("MAX_HISTORY_CHARS", "6000"))


# -----------------------
# In-process memory store (per session_id)
# If you need persistence across restarts, move this to Redis.
# -----------------------
SESSION_MEMORIES: Dict[str, ConversationBufferMemory] = {}


def get_memory(session_id: str) -> ConversationBufferMemory:
    if session_id not in SESSION_MEMORIES:
        SESSION_MEMORIES[session_id] = ConversationBufferMemory(
            return_messages=True,
            memory_key="chat_history",
            input_key="question",
            output_key="answer",
        )
    return SESSION_MEMORIES[session_id]


def render_history(memory: ConversationBufferMemory) -> str:
    """
    Convert last N turns to a compact string (bounded by chars).
    """
    msgs = memory.chat_memory.messages
    # Keep last MAX_HISTORY_TURNS*2 messages (human+ai)
    msgs = msgs[-(MAX_HISTORY_TURNS * 2):]

    lines = []
    for m in msgs:
        if isinstance(m, HumanMessage):
            lines.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            lines.append(f"Assistant: {m.content}")
        else:
            # fallback
            lines.append(f"{m.type}: {getattr(m, 'content', '')}")

    text = "\n".join(lines).strip()
    if len(text) > MAX_HISTORY_CHARS:
        text = text[-MAX_HISTORY_CHARS:]  # keep last chars
    return text


# -----------------------
# Redis helpers
# -----------------------
def redis_client() -> redis.Redis:
    return redis.Redis.from_url(REDIS_URL, decode_responses=True)

def _stable_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def _hash_key(*parts: Any) -> str:
    blob = "|".join(_stable_json(p) if isinstance(p, (dict, list, tuple)) else str(p) for p in parts)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()

def cache_get(r: redis.Redis, key: str) -> Optional[Any]:
    v = r.get(key)
    if v is None:
        return None
    try:
        return json.loads(v)
    except Exception:
        return None

def cache_set(r: redis.Redis, key: str, value: Any, ttl: int) -> None:
    r.setex(key, ttl, json.dumps(value, ensure_ascii=False))


# -----------------------
# OpenSearch client
# -----------------------
def connect_opensearch() -> OpenSearch:
    kwargs = dict(
        hosts=[{"host": OS_HOST, "port": OS_PORT}],
        http_auth=(OS_USER, OS_PASS),
        use_ssl=USE_SSL,
        verify_certs=VERIFY_CERTS,
        ssl_show_warn=SSL_SHOW_WARN,
        timeout=60,
        max_retries=3,
        retry_on_timeout=True,
    )
    if USE_SSL and VERIFY_CERTS and CA_CERTS:
        kwargs["ca_certs"] = CA_CERTS
    return OpenSearch(**kwargs)


# -----------------------
# Retrieval
# -----------------------
def bm25_search(client: OpenSearch, query: str, k: int, filters: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    must_filters = []
    if filters:
        for field, value in filters.items():
            must_filters.append({"term": {field: value}})

    body = {
        "size": k,
        "_source": [TEXT_FIELD, "source", "title", "chunk_index", "metadata"],
        "query": {
            "bool": {
                "must": [{"match": {TEXT_FIELD: {"query": query}}}],
                "filter": must_filters
            }
        }
    }
    resp = client.search(index=INDEX_NAME, body=body)
    return resp.get("hits", {}).get("hits", [])


def knn_search(client: OpenSearch, query_vector: List[float], k: int, filters: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    must_filters = []
    if filters:
        for field, value in filters.items():
            must_filters.append({"term": {field: value}})

    body = {
        "size": k,
        "_source": [TEXT_FIELD, "source", "title", "chunk_index", "metadata"],
        "query": {
            "bool": {
                "filter": must_filters,
                "must": [
                    {
                        "knn": {
                            VECTOR_FIELD: {
                                "vector": query_vector,
                                "k": k
                            }
                        }
                    }
                ]
            }
        }
    }
    resp = client.search(index=INDEX_NAME, body=body)
    return resp.get("hits", {}).get("hits", [])


def rrf_fuse(bm25_hits: List[Dict[str, Any]], knn_hits: List[Dict[str, Any]], top_n: int, k_rrf: int = 60) -> List[Tuple[Dict[str, Any], float]]:
    scores: Dict[str, Dict[str, Any]] = {}

    def add(hits: List[Dict[str, Any]], weight: float = 1.0):
        for rank, h in enumerate(hits, start=1):
            doc_id = h["_id"]
            if doc_id not in scores:
                scores[doc_id] = {"hit": h, "score": 0.0}
            scores[doc_id]["score"] += weight * (1.0 / (k_rrf + rank))

    add(bm25_hits, 1.0)
    add(knn_hits, 1.0)

    fused = [(v["hit"], float(v["score"])) for v in scores.values()]
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:top_n]


def build_context(fused_hits: List[Tuple[Dict[str, Any], float]], max_chars: int = 12000) -> str:
    parts = []
    total = 0
    for hit, score in fused_hits:
        src = hit.get("_source", {}) or {}
        text = (src.get(TEXT_FIELD) or "").strip()
        if not text:
            continue
        source_name = src.get("source") or (src.get("metadata", {}) or {}).get("source", "unknown")
        chunk_index = src.get("chunk_index", (src.get("metadata", {}) or {}).get("chunk_index", ""))
        header = f"[{source_name} | chunk {chunk_index} | rrf={score:.4f}]"
        block = header + "\n" + text + "\n"
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n---\n".join(parts)


# -----------------------
# Caching: embedding + retrieval (+ optional answer)
# -----------------------
def embed_query_cached(r: redis.Redis, embeddings: OpenAIEmbeddings, question: str) -> List[float]:
    key = f"{REDIS_PREFIX}:emb:{_hash_key(EMBED_MODEL, question)}"
    if CACHE_EMBEDDING:
        cached = cache_get(r, key)
        if isinstance(cached, list) and cached:
            return cached
    vec = embeddings.embed_query(question)
    if CACHE_EMBEDDING:
        cache_set(r, key, vec, TTL_EMBEDDING)
    return vec


def retrieve_hybrid_cached(
    r: redis.Redis,
    client: OpenSearch,
    question: str,
    qvec: List[float],
    bm25_k: int,
    knn_k: int,
    filters: Optional[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    base = {"index": INDEX_NAME, "q": question, "bm25_k": bm25_k, "knn_k": knn_k, "filters": filters or {}}
    bm25_key = f"{REDIS_PREFIX}:bm25:{_hash_key(base)}"
    knn_key = f"{REDIS_PREFIX}:knn:{_hash_key(base)}"

    if CACHE_RETRIEVAL:
        bm25_cached = cache_get(r, bm25_key)
        knn_cached = cache_get(r, knn_key)
        if isinstance(bm25_cached, list) and isinstance(knn_cached, list):
            return bm25_cached, knn_cached

    bm25_hits = bm25_search(client, question, bm25_k, filters=filters)
    knn_hits = knn_search(client, qvec, knn_k, filters=filters)

    if CACHE_RETRIEVAL:
        cache_set(r, bm25_key, bm25_hits, TTL_RETRIEVAL)
        cache_set(r, knn_key, knn_hits, TTL_RETRIEVAL)

    return bm25_hits, knn_hits


# -----------------------
# LLM with both: history + retrieved context
# -----------------------
def answer_with_openai(question: str, history: str, retrieved_context: str) -> str:
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    prompt = (
        "You are a precise assistant.\n"
        "Use BOTH the conversation history and the retrieved documents.\n"
        "If the retrieved documents do not contain enough information, say:\n"
        "'I don't know based on the provided documents.'\n"
        "Do not invent facts.\n\n"
        f"Conversation history:\n{history or '(none)'}\n\n"
        f"Retrieved documents:\n{retrieved_context or '(none)'}\n\n"
        f"User question:\n{question}\n"
    )
    return llm.invoke(prompt).content


# -----------------------
# Main RAG function (with memory)
# -----------------------
def hybrid_rag_answer(
    session_id: str,
    question: str,
    bm25_k: int = 20,
    knn_k: int = 20,
    fuse_top_n: int = 8,
    filters: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    r = redis_client()
    client = connect_opensearch()
    memory = get_memory(session_id)

    # Optional answer cache: include history digest so cache is correct per conversation state
    history_text = render_history(memory)
    ans_key_payload = {
        "session_id": session_id,
        "q": question,
        "history_digest": hashlib.sha256(history_text.encode("utf-8")).hexdigest(),
        "bm25_k": bm25_k,
        "knn_k": knn_k,
        "top_n": fuse_top_n,
        "filters": filters or {},
        "chat_model": CHAT_MODEL,
        "embed_model": EMBED_MODEL,
    }
    answer_key = f"{REDIS_PREFIX}:ans:{_hash_key(ans_key_payload)}"

    if CACHE_ANSWER:
        cached = cache_get(r, answer_key)
        if isinstance(cached, dict) and "answer" in cached:
            # also append to memory so the conversation continues naturally
            memory.chat_memory.add_user_message(question)
            memory.chat_memory.add_ai_message(cached["answer"])
            cached["cached"] = True
            return cached

    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    qvec = embed_query_cached(r, embeddings, question)

    bm25_hits, knn_hits = retrieve_hybrid_cached(
        r=r,
        client=client,
        question=question,
        qvec=qvec,
        bm25_k=bm25_k,
        knn_k=knn_k,
        filters=filters,
    )

    fused = rrf_fuse(bm25_hits, knn_hits, top_n=fuse_top_n, k_rrf=60)
    retrieved_context = build_context(fused)

    # history updated just-in-time (exclude current question so history reflects previous turns)
    history_text = render_history(memory)

    #answer = answer_with_openai(question, history_text, retrieved_context)
    answer = answer_with_openai(question, '', retrieved_context)

    # Update memory
    memory.chat_memory.add_user_message(question)
    memory.chat_memory.add_ai_message(answer)

    sources = []
    for hit, score in fused:
        src = hit.get("_source", {}) or {}
        sources.append({
            "id": hit.get("_id"),
            "source": src.get("source") or (src.get("metadata", {}) or {}).get("source"),
            "chunk_index": src.get("chunk_index", (src.get("metadata", {}) or {}).get("chunk_index")),
            "rrf_score": score,
        })

    out = {
        "session_id": session_id,
        "answer": answer,
        "sources": sources,
        "bm25_count": len(bm25_hits),
        "knn_count": len(knn_hits),
        "cached": False,
    }

    if CACHE_ANSWER:
        cache_set(r, answer_key, out, TTL_ANSWER)

    return out


# -----------------------
# CLI
# -----------------------
# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--session_id", required=True, help="Conversation session id (e.g. user123)")
#     parser.add_argument("--q", required=True, help="Question to ask")
#     parser.add_argument("--bm25_k", type=int, default=20)
#     parser.add_argument("--knn_k", type=int, default=20)
#     parser.add_argument("--top_n", type=int, default=8)
#     parser.add_argument("--filter_source", default=None, help="Optional: filter by source filename")
#     args = parser.parse_args()

#     filters = None
#     if args.filter_source:
#         filters = {"source": args.filter_source}

#     out = hybrid_rag_answer(
#         session_id=args.session_id,
#         question=args.q,
#         bm25_k=args.bm25_k,
#         knn_k=args.knn_k,
#         fuse_top_n=args.top_n,
#         filters=filters,
#     )

#     print("\nSESSION:", out["session_id"])
#     print("\nANSWER:\n", out["answer"])
#     print("\nCACHED:", out["cached"])
#     print("\nSOURCES:")
#     for s in out["sources"]:
#         print(s)

In [16]:
out = hybrid_rag_answer(
        session_id="user123",
        question="Explain what are states in this state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'session_id': 'user123', 'answer': "Based on the provided documents, here are the states in the finite-state machine (FSM) implementation:\n\n- **Relevance** \n  - *Purpose: This state determines the domain relevance of the query. If the query is deemed irrelevant, the process may terminate early.*\n\n- **Confidence** \n  - *Purpose: This state scores the answerability of the query using a strict JSON format and generates a fast-draft snippet for reference.*\n\n- **Decomposition** \n  - *Purpose: This state breaks the question into 2–3 concrete subtopics, such as device physics, system impact, or implementation.*\n\n- **Self Evaluation** \n  - *Purpose: In this state, the system evaluates its confidence for each subtopic using discrete confidence levels (0, 0.25, 0.5, 0.75, 1) to decide whether to search online or proceed with answering.*\n\n- **Knowledge** \n  - *Purpose: This state compares different models to quantify accuracy versus cost and ensures that the information retrieved 

In [11]:
out = hybrid_rag_answer(
        session_id="user123",
        question="Explain what are states in this state machine implementation",
        bm25_k=20,
        knn_k=20,
        fuse_top_n=8,
        filters=None,
    )
print(out)

{'session_id': 'user123', 'answer': "Based on the provided documents, here are the key points regarding the states in the finite-state machine (FSM) implementation:\n\n- **Relevance State**  \n  - *Purpose*: This state is responsible for determining the relevance of the input query. It may terminate the process early if the query is deemed irrelevant.\n\n- **Confidence State**  \n  - *Purpose*: In this state, the system scores the answerability of the query using a strict JSON format. It also generates a fast-draft snippet for reference, which helps in assessing the confidence level of the response.\n\n- **Decomposition State**  \n  - *Purpose*: This state breaks down the original question into 2–3 concrete subtopics, such as device physics, system impact, or implementation, to facilitate more focused retrieval and answering.\n\n- **Self-Evaluation State**  \n  - *Purpose*: Here, the system evaluates its confidence for each subtopic using discrete confidence levels (0, 0.25, 0.5, 0.75,

In [3]:
!pip show langchain

Name: langchain
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
